# Location Selection with E-NAUTILUS: Part 2
_Running of E-NAUTILUS for decision making, and presentation of results_


In [59]:
import numpy as np
import pandas as pd
import polars as pl
import pickle
import folium
# These are to just suppress warnings in the outputs of the example
import warnings

warnings.filterwarnings("ignore")

## Load results from previous session

In [60]:
file_name = "data/scenarios/pf_16_15minute.pkl"

output = open(file_name, 'rb')
prev_session = pickle.load(output)

raw_ref_pf = prev_session["pf"]
prob = prev_session["prob"]
events = prev_session["events"]
cities = prev_session["cities"]
event2city = prev_session["event2city"]


## Load reference front and problem 

In [61]:
output_flat = np.array(raw_ref_pf).flatten()

def process_lists(dict2conv):
    return {key: np.array(dict2conv[key]).flatten().tolist() for key in dict2conv.keys()}

# TODO include constraints in here too
output_dict = [
    output.optimal_objectives | 
    process_lists(output.optimal_variables) 
    for output in output_flat]

nd_df = pl.DataFrame(output_dict)

nd_df = nd_df.unique(subset=("f_1", "f_2", "f_3", "f_4"))

nd_df = nd_df.with_columns([
    (-pl.col("f_1")).alias("f_1_min"),
    (pl.col("f_2")).alias("f_2_min"),
    (pl.col("f_3")).alias("f_3_min"),
    (-pl.col("f_4")).alias("f_4_min")
])

nadir_point = {
  "f_1": float(nd_df["f_1"].min()),
  "f_2": float(nd_df["f_2"].max()),
  "f_3": float(nd_df["f_3"].max()),
  "f_4": float(nd_df["f_4"].min())
}

display(nd_df)

print(f"Nadir point: {nadir_point}")
print(f"Nadir point (problem): {prob.get_nadir_point()}")
print(f"Idedal point (problem): {prob.get_ideal_point()}")


reachable_indices = list(range(len(nd_df)))  # everything reachable from nadir


f_1,f_2,f_3,f_4,ev,cover,_alpha,f_1_min,f_2_min,f_3_min,f_4_min
f64,f64,f64,f64,list[f64],list[f64],list[f64],f64,f64,f64,f64
502.0,4.0,1146.675,0.790439,"[1.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.210526],-502.0,4.0,1146.675,-0.790439
0.0,0.0,0.0,0.0,"[0.0, 0.0, … 0.0]","[0.0, 0.0, … 0.0]",[-9.9600e-9],-0.0,0.0,0.0,-0.0
495.0,2.0,1010.545,0.611479,"[1.0, 0.0, … 0.0]","[1.0, 1.0, … 0.0]",[0.128899],-495.0,2.0,1010.545,-0.611479
404.0,2.0,853.265,0.691074,"[1.0, 0.0, … 0.0]","[1.0, 1.0, … 0.0]",[-0.68948],-404.0,2.0,853.265,-0.691074
16.0,0.0,110.565,0.070937,"[0.0, 0.0, … 0.0]","[0.0, 0.0, … 0.0]",[-9.9900e-9],-16.0,0.0,110.565,-0.070937
…,…,…,…,…,…,…,…,…,…,…
487.0,3.0,1007.8,0.636097,"[1.0, 0.0, … 0.0]","[1.0, 1.0, … 0.0]",[0.366758],-487.0,3.0,1007.8,-0.636097
10.0,1.0,76.0,0.038295,"[1.0, 0.0, … 0.0]","[1.0, 1.0, … 0.0]",[0.027658],-10.0,1.0,76.0,-0.038295
404.0,2.0,853.265,0.698465,"[1.0, 0.0, … 0.0]","[1.0, 1.0, … 0.0]",[0.31052],-404.0,2.0,853.265,-0.698465


Nadir point: {'f_1': 0.0, 'f_2': 19.0, 'f_3': 2747.86, 'f_4': 0.0}
Nadir point (problem): {'f_1': 0, 'f_2': 19, 'f_3': 2747.86, 'f_4': 0}
Idedal point (problem): {'f_1': 565, 'f_2': 0, 'f_3': 0, 'f_4': 1.0}


## Helper functions

In [62]:
# Function to determine marker size based on population
def get_marker_size(population):
    return max(5, population / 1000)  # Adjust the divisor to scale marker size

def create_color_dict(cities, ev_cities, cc): 
    marker_color = {}
    for city in cities.loc[:,"city"]: 
        if city in ev_cities: 
            marker_color[city] = "orange"
        elif city in cc: 
            marker_color[city] = "yellow"
        else: 
            marker_color[city] = "grey"

    return marker_color

def select_point(results, sol_id): 

    return {
        "f_1": int(results.loc[sol_id, "Total patients served"]),
        "f_2": int(results.loc[sol_id, "Number of overstaffed events"]),
        "f_3": float(results.loc[sol_id, "Total costs ($)"]),
        "f_4": float(results.loc[sol_id, "Population with access (%)"]/100.0)
        }


def clean_results(raw_results, intermediate_point=True): 
    # Transform objectives
    if intermediate_point: 
        results = pd.DataFrame(raw_results.intermediate_points)
    else:
        results = pd.DataFrame(raw_results.optimal_objectives)

    results = results.rename(columns={
                    "f_1": "Total patients served", 
                    "f_2": "Number of overstaffed events", 
                    "f_3": "Total costs ($)", 
                    "f_4": "Population with access (%)"})
    results[["Total patients served", "Number of overstaffed events"]] =  results[["Total patients served", "Number of overstaffed events"]].astype(int)
    results[["Population with access (%)"]] = (results[["Population with access (%)"]]*100.0).round(2)
    results[["Total costs ($)"]] = (results[["Total costs ($)"]]).round(2)

    results.index.name = "Solution ID"

    return results

## Run eNAUTILUS 
### Round 1
We're going to generate some solutions. They will be poor at first, but you and the computer will slowly find the best solution that fulfills your goals and preferences. 


In [63]:
# Initialize a first solution 
from desdeo.mcdm.enautilus import enautilus_step
from desdeo.mcdm.enautilus import enautilus_get_representative_solutions

current_iter = 0
selected_point = nadir_point
total_iterations = 3
display(f"Starting with point {selected_point}")

prob.get_ideal_point()


raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you prefer?")
display(results)

"Starting with point {'f_1': 0.0, 'f_2': 19.0, 'f_3': 2747.86, 'f_4': 0.0}"

number of iterations left: 3


'Which solution to do you prefer?'

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,3,13,1857.24,1.28
1,162,13,2167.84,21.20
2,188,19,2747.86,26.00


### Round 2 

In [64]:
chosen_solution = 2

In [65]:
# Refine solution based on feedback
current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")
display(raw_results)
results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)
display("Results:")



{'f_1': 188, 'f_2': 19, 'f_3': 2747.86, 'f_4': 0.26}
number of iterations left: 2


ENautilusResult(current_iteration=2, iterations_left=1, intermediate_points=[{'f_1': 99.0, 'f_2': 10.0, 'f_3': 1411.93, 'f_4': 0.1491477082324906}, {'f_1': 337.5, 'f_2': 11.0, 'f_3': 1877.83, 'f_4': 0.4480486775638878}, {'f_1': 376.5, 'f_2': 19.0, 'f_3': 2747.86, 'f_4': 0.5199355847637488}], reachable_best_bounds=[{'f_1': 511.0, 'f_2': 1.0, 'f_3': 643.545, 'f_4': 0.8491144456442145}, {'f_1': 515.0, 'f_2': 1.0, 'f_3': 643.545, 'f_4': 0.8596827642700227}, {'f_1': 565.0, 'f_2': 1.0, 'f_3': 853.265, 'f_4': 0.8596827642700227}], reachable_worst_bounds=[{'f_1': 99.0, 'f_2': 10.0, 'f_3': 1411.93, 'f_4': 0.1491477082324906}, {'f_1': 337.5, 'f_2': 11.0, 'f_3': 1877.83, 'f_4': 0.4480486775638878}, {'f_1': 376.5, 'f_2': 19.0, 'f_3': 2747.86, 'f_4': 0.5199355847637488}], closeness_measures=[50.13674627312301, 51.645516394727075, 66.63716820305491], reachable_point_indices=[[0, 2, 3, 6, 7, 8, 9, 12, 14, 15, 17, 18], [0, 2, 3, 5, 6, 7, 8, 9, 12, 14, 15, 17, 18, 19], [0, 2, 3, 5, 6, 7, 8, 10, 12, 14,

'Which solution to do you find most preferable?'

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,99,10,1411.93,14.91
1,337,11,1877.83,44.80
2,376,19,2747.86,51.99


'Results:'

### Round 3

In [66]:
chosen_solution = 2

In [67]:
# Refine solution based on feedback
current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)




{'f_1': 376, 'f_2': 19, 'f_3': 2747.86, 'f_4': 0.5199}
number of iterations left: 1


'Which solution to do you find most preferable?'

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,10,1,76.00,3.83
1,487,3,1007.80,63.61
2,565,19,2747.86,77.99


## Display final result

In [ ]:
final_chosen_solution = 2

In [55]:
# Get final solution 
solutions = enautilus_get_representative_solutions(prob, raw_results, nd_df) 
solution = solutions[final_chosen_solution]
results = clean_results(solution, intermediate_point=False)

display(results)
 


,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,487,3,1007.8,63.61


### Result postprocessing

In [56]:
# Post process result...
raw_events = solution.optimal_variables['ev'][0].to_list()
raw_events = [[bool(e) for e in raw_events]]

raw_coverage = solution.optimal_variables['cover'][0].to_list()
raw_coverage = [[bool(c) for c in raw_coverage]]

events_visited = []
for evb in raw_events: 
    events_visited.append("\n".join(events.loc[evb, "event_id"].values))

cities_covered = [] 
for cc in raw_coverage: 
    cities_covered.append("\n".join(cities.loc[cc,"city"].values))

results["Events Visited"] = events_visited
results["Cities covered"] = cities_covered

results

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%),Events Visited,Cities covered
Solution ID,,,,,,
0,487,3,1007.8,63.61,ada-public-library\nbluffton-bluffton-public-l...,Ada\nAlger\nBluffton\nCairo\nCridersville\nDel...


### Map preprocessing

In [57]:
# Process for the map
events_in_cities = events.loc[raw_events[0],:].groupby("city").agg({"event_pretty": lambda e : '<br>'.join(e)})
events_in_cities = events_in_cities.to_dict()['event_pretty']
events_in_cities

# cc cities covered
cc = set(cities_covered[0].split('\n'))

# event_cities 
ev_cities = list(events.loc[raw_events[0], "city"])
marker_colors = create_color_dict(cities, ev_cities, cc)


## Event coverage dictionary 
event2city_mat = event2city[raw_events[0]].astype(bool)

event2city_dict = {}
for (c,city) in enumerate(ev_cities): 
    event2city_dict[city] = set(cities.loc[event2city_mat[c,],"city"]) - {city}

adjacent_events = {}
# Record what events are near cities
for event_city in event2city_dict.keys(): 
    adj_cities = event2city_dict[event_city]
    from_name = cities.loc[cities.loc[:,"city"] == event_city,["city"]].values.tolist()[0][0]

    for adj_city in adj_cities: 
        to_name = cities.loc[cities.loc[:,"city"] == adj_city ,["city"]].values.tolist()[0][0]

        if to_name not in adjacent_events.keys(): 
            adjacent_events[to_name] = {from_name}
        else: 
            adjacent_events[to_name] = adjacent_events[to_name].union({from_name})



## Render map

In [58]:
# Render map

# Create a base map
m = folium.Map(location=[cities['lat'].mean(), 
                         cities['long'].mean()], 
                         zoom_start=7) 

# Draw lines between 
for event_city in event2city_dict.keys(): 
    adj_cities = event2city_dict[event_city]
    from_loc = cities.loc[cities.loc[:,"city"] == event_city,["lat", "long"]].values.tolist()

    for adj_city in adj_cities: 
        to_loc = cities.loc[cities.loc[:,"city"] == adj_city ,["lat", "long"]].values.tolist()
        folium.PolyLine(
            locations=[to_loc[0], from_loc[0]],
            color="black"
        ).add_to(m)

# Set bounds
sw = cities.loc[:,['lat', 'long']].min().values.tolist()
ne = cities.loc[:,['lat', 'long']].max().values.tolist()
m.fit_bounds([sw,ne])

# Create tool tips 
tooltips = {}
for _, row in cities.iterrows():
    city = row['city']
    tooltips[city]=f"<b>{city}</b><br><b>Population:</b> {row['pop']}"

    if city in events_in_cities.keys():
        tooltips[city]+= "<br><b>Events:</b><br>"
        tooltips[city]+= events_in_cities[city]
    else:
        tooltips[city]+= "<br><b>No Healthwise Clinics</b>"

    if city in adjacent_events.keys(): 
        tooltips[city]+= "<br><b>Covered by events in: </b>"
        tooltips[city]+= "<br>".join(adjacent_events[city])


# Add cities to the map
for _, row in cities.iterrows():
    city = row['city']
    folium.CircleMarker(
        location=(row['lat'], row['long']),
        radius=get_marker_size(row['pop']),
        color="black",
        fill=True,
        fill_color=marker_colors[city],
        fill_opacity=0.6,
        tooltip=tooltips[city]
    ).add_to(m)

display(results)
display(m)

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%),Events Visited,Cities covered
Solution ID,,,,,,
0,487,3,1007.8,63.61,ada-public-library\nbluffton-bluffton-public-l...,Ada\nAlger\nBluffton\nCairo\nCridersville\nDel...
